# JSONBench walkthrough

Generate a small travel dataset in memory.

In [ ]:
from json_bench.data import generate_travel_dataset

dataset = generate_travel_dataset(
    scale=0.01,
    seed=23,
    reference_date="2025-11-01",
)

## Deploy MongoDB locally

Start Docker first. BONDI creates a named local deployment and waits until MongoDB is ready.
This example uses port 27127 to avoid the existing JSON-Storm deployment.
Change the name and port if needed.

In [ ]:
from bondi.deployment import deploy_mongodb

deployment = deploy_mongodb(name="jsonbench-travel", port=27127)
print(deployment.uri)

## Load the travel collections

Call BONDI once per collection. Each reference specifies the database and collection name.
The loader preserves the generated `_id` values and date strings.
Its default `create` mode requires an empty collection. To repeat the walkthrough,
run cleanup first, then rerun deployment and loading.

In [ ]:
from bondi.data import load_documents
from bondi.mongodb import MongoCollectionRef

for name, documents in dataset.items():
    result = load_documents(
        documents,
        target=deployment,
        collection=MongoCollectionRef("travel", name),
    )
    print(result.destinations)

## Deploy Couchbase locally

Run `uv sync --locked` to install the Couchbase SDK added to this project.
BONDI creates a local Community Edition deployment with a `travel` bucket.
These ports are separate from MongoDB and the existing JSON-Storm deployment.
Change the name and ports if needed.

In [ ]:
from bondi.deployment import deploy_couchbase

couchbase_deployment = deploy_couchbase(
    name="jsonbench-travel-cb",
    management_port=18291,
    query_port=18293,
    kv_port=21230,
    bucket="travel",
)
print(couchbase_deployment.console_url)

## Load the same dataset into Couchbase

Reuse `dataset` from the generation cell. Each reference names a bucket, scope and collection.
BONDI creates the four collections in the `_default` scope.
The document bodies retain their `_id` values and ISO date strings.
BONDI derives Couchbase document keys from the IDs. No query indexes are created.
The default `create` mode requires empty collections, as with MongoDB.

In [ ]:
from bondi.couchbase import CouchbaseCollectionRef

for name, documents in dataset.items():
    result = load_documents(
        documents,
        target=couchbase_deployment,
        collection=CouchbaseCollectionRef("travel", "_default", name),
    )
    print(result.destinations)

## Clean up when finished

Each cell removes one notebook deployment, including its data and saved configuration.
Run the relevant cell when finished, or if loading fails after deployment succeeds.

In [ ]:
deployment.remove()

In [ ]:
couchbase_deployment.remove()